In [1]:
import requests
import pandas as pd
import time
from pathlib import Path

from IPython.display import clear_output

In [2]:
PROVINCES = {
    "10": "กรุงเทพมหานคร",
    "11": "สมุทรปราการ",
    "12": "นนทบุรี",
    "13": "ปทุมธานี",
    "14": "พระนครศรีอยุธยา",
    "15": "อ่างทอง",
    "16": "ลพบุรี",
    "17": "สิงห์บุรี",
    "18": "ชัยนาท",
    "19": "สระบุรี",
    "20": "ชลบุรี",
    "21": "ระยอง",
    "22": "จันทบุรี",
    "23": "ตราด",
    "24": "ฉะเชิงเทรา",
    "25": "ปราจีนบุรี",
    "26": "นครนายก",
    "27": "สระแก้ว",
    "30": "นครราชสีมา",
    "31": "บุรีรัมย์",
    "32": "สุรินทร์",
    "33": "ศรีสะเกษ",
    "34": "อุบลราชธานี",
    "35": "ยโสธร",
    "36": "ชัยภูมิ",
    "37": "อำนาจเจริญ",
    "38": "บึงกาฬ",
    "39": "หนองบัวลำภู",
    "40": "ขอนแก่น",
    "41": "อุดรธานี",
    "42": "เลย",
    "43": "หนองคาย",
    "44": "มหาสารคาม",
    "45": "ร้อยเอ็ด",
    "46": "กาฬสินธุ์",
    "47": "สกลนคร",
    "48": "นครพนม",
    "49": "มุกดาหาร",
    "50": "เชียงใหม่",
    "51": "ลำพูน",
    "52": "ลำปาง",
    "53": "อุตรดิตถ์",
    "54": "แพร่",
    "55": "น่าน",
    "56": "พะเยา",
    "57": "เชียงราย",
    "58": "แม่ฮ่องสอน",
    "60": "นครสวรรค์",
    "61": "อุทัยธานี",
    "62": "กำแพงเพชร",
    "63": "ตาก",
    "64": "สุโขทัย",
    "65": "พิษณุโลก",
    "66": "พิจิตร",
    "67": "เพชรบูรณ์",
    "70": "ราชบุรี",
    "71": "กาญจนบุรี",
    "72": "สุพรรณบุรี",
    "73": "นครปฐม",
    "74": "สมุทรสาคร",
    "75": "สมุทรสงคราม",
    "76": "เพชรบุรี",
    "77": "ประจวบคีรีขันธ์",
    "80": "นครศรีธรรมราช",
    "81": "กระบี่",
    "82": "พังงา",
    "83": "ภูเก็ต",
    "84": "สุราษฎร์ธานี",
    "85": "ระนอง",
    "86": "ชุมพร",
    "90": "สงขลา",
    "91": "สตูล",
    "92": "ตรัง",
    "93": "พัทลุง",
    "94": "ปัตตานี",
    "95": "ยะลา",
    "96": "นราธิวาส"
}

In [3]:
URL = "https://opendata.moph.go.th/api/report_data"


def get_moph_data(
    table_name,
    year,
    province,
    limit=1000,
    max_retries=5,
    base_wait=2
):
    all_rows = []
    offset = 0
    page = 1

    while True:

        payload = {
            "tableName": table_name,
            "year": int(year),
            "province": int(province),
            "type": "json",
            "limit": limit,
            "offset": offset
        }

        response = None

        for attempt in range(max_retries + 1):

            try:
                response = requests.post(
                    URL,
                    json=payload,
                    timeout=60
                )

            except requests.RequestException as e:

                if attempt >= max_retries:
                    print(
                        f"REQUEST FAILED | "
                        f"{table_name} | {year} | {province} | "
                        f"offset={offset} | {e}"
                    )
                    return None

                wait_time = base_wait * (2 ** attempt)

                print(
                    f"REQUEST ERROR | retry {attempt + 1}/{max_retries} "
                    f"in {wait_time}s | "
                    f"{table_name} | {year} | {province}"
                )

                time.sleep(wait_time)
                continue

            # --------------------------------------
            # 429 Too Many Requests
            # --------------------------------------
            if response.status_code == 429:

                if attempt >= max_retries:
                    print(
                        f"RATE LIMIT FAILED | "
                        f"{table_name} | {year} | {province} | "
                        f"offset={offset}"
                    )
                    return None

                retry_after = response.headers.get("Retry-After")

                if retry_after:
                    try:
                        wait_time = float(retry_after)
                    except ValueError:
                        wait_time = base_wait * (2 ** attempt)
                else:
                    wait_time = base_wait * (2 ** attempt)

                print(
                    f"429 TOO MANY REQUESTS | "
                    f"retry {attempt + 1}/{max_retries} "
                    f"in {wait_time:.1f}s | "
                    f"{table_name} | {year} | {province} | "
                    f"offset={offset:,}"
                )

                time.sleep(wait_time)
                continue

            # --------------------------------------
            # 400 Bad Request -> skip
            # --------------------------------------
            if response.status_code == 400:
                print(
                    f"BAD REQUEST | "
                    f"{table_name} | {year} | {province}"
                )
                return None

            # --------------------------------------
            # Retry server errors
            # --------------------------------------
            if response.status_code in [500, 502, 503, 504]:

                if attempt >= max_retries:
                    print(
                        f"SERVER ERROR {response.status_code} | "
                        f"FAILED | "
                        f"{table_name} | {year} | {province}"
                    )
                    return None

                wait_time = base_wait * (2 ** attempt)

                print(
                    f"SERVER ERROR {response.status_code} | "
                    f"retry {attempt + 1}/{max_retries} "
                    f"in {wait_time}s"
                )

                time.sleep(wait_time)
                continue

            # other HTTP errors
            response.raise_for_status()

            # successful request
            break

        # ------------------------------------------
        # Parse JSON
        # ------------------------------------------
        try:
            result = response.json()

        except ValueError:
            print(
                f"INVALID JSON | "
                f"{table_name} | {year} | {province} | "
                f"offset={offset:,}"
            )
            return None

        if isinstance(result, list):

            temp = pd.DataFrame(result)

        elif isinstance(result, dict):

            if "data" in result:
                temp = pd.DataFrame(result["data"])

            else:
                temp = pd.DataFrame([result])

        else:

            print(
                f"UNKNOWN RESPONSE | "
                f"{table_name} | {year} | {province}"
            )

            break

        n_rows = len(temp)

        print(
            f"OK | "
            f"{table_name} | "
            f"{year} | "
            f"{province} | "
            f"page={page} | "
            f"offset={offset:,} | "
            f"rows={n_rows:,}"
        )

        # no more records
        if temp.empty:
            break

        all_rows.append(temp)

        # final page
        if n_rows < limit:
            break

        offset += limit
        page += 1

        # gentle delay between normal requests
        time.sleep(0.2)

    if not all_rows:
        return None

    return pd.concat(
        all_rows,
        ignore_index=True
    )

In [4]:
YEARS = range(2565, 2569)

# TABLES = [
#     "s_dm_pop_age",
#     "s_ht_pop_age",
#     "s_copd_pop_age",
#     "s_coronary_pop_age",
#     "s_stroke_pop_age",
#     "s_emphysema_pop_age",
#     "s_ca_breast_pop_age",
#     "s_ca_lung_pop_age",
#     "s_ca_cervix_pop_age",
#     "s_dental_2",
#     "s_dental_4",
#     "s_pharm1",
#     "s_pharm2",
#     "s_palliative_com",
#     "s_rop1",
#     "s_rop2",
#     "s_ckd_stage_typearea",
#     "s_ckd_stage_hosp"
# ]

TABLES = [
    "s_ht_control_new",
    "s_dm_control"
]


output_dir = Path("output/scraped")
output_dir.mkdir(exist_ok=True)

data_by_table = {}

for table_name in TABLES:

    print(f"\n===== {table_name} =====")

    table_data = []

    for year in YEARS:

        for province_code, province_name in PROVINCES.items():

            df = get_moph_data(
                table_name=table_name,
                year=year,
                province=province_code,
                limit=1000
            )

            if df is None or df.empty:
                continue

            df["query_table"] = table_name
            df["query_year"] = year
            df["query_province_code"] = province_code
            df["query_province_name"] = province_name

            table_data.append(df)

    if table_data:
        scrp_df = pd.concat(
            table_data,
            ignore_index=True
        )

        scrp_df.to_csv(
            output_dir / f"{table_name}.csv",
            index=False,
            encoding="utf-8-sig"
        )

    else:
        scrp_df = pd.DataFrame()

    clear_output()

In [ ]:
Pivot year and prov
Sum col pop_group 1,2,3,4,5
Sum col result_group 1,2,3,4,5
Calculate the percentage as prevalence
    "s_dm_pop_age",
    "s_ht_pop_age",
    "s_copd_pop_age",
    "s_coronary_pop_age",
    "s_stroke_pop_age",
    "s_emphysema_pop_age",
    "s_ca_breast_pop_age",
    "s_ca_lung_pop_age",
    "s_ca_cervix_pop_age",

Pivot year and prov
    "s_dental_2",
    "s_dental_4",

Pivot year and prov
    "s_pharm1", Sum col begin with r1m...
    "s_pharm2", Sum col begin with r2m...

Pivot year and prov
Sum target1, result1
    "s_palliative_com",

Pivot year and prov
    "s_ckd_stage_typearea",
    "s_ckd_stage_hosp"